# Reddit pilot: audited data to Independent Leiden

This notebook replays **real recorded results**, verifies their provenance, and produces eight descriptive figure families. Tracking, features and forecasting evaluation are not implemented. Subreddit names describe activity contexts, not ground-truth user communities.

## Setup and kernel (run once in a terminal at the repository root)
Python **3.11+** is required. In Windows PowerShell:
```powershell
python -m venv .venv
& .\.venv\Scripts\python.exe -m pip install -e '.[graph,notebook]'
& .\.venv\Scripts\python.exe -m ipykernel install --prefix .venv --name reddit-sna --display-name 'Reddit SNA (.venv)'
```
In VS Code, install the Python and Jupyter extensions, click **Select Kernel > Python Environments** and choose this repository's `.venv\Scripts\python.exe`. Restart the kernel after installing dependencies. In Jupyter, select `Reddit SNA (.venv)` if available; otherwise select the same interpreter. Linux/macOS use `.venv/bin/python`.

Run **Restart Kernel > Run All**. The first cell prints the interpreter and gives actionable setup errors. No cell installs packages or edits `sys.path`. The `00_research_plan.ipynb` files are unfinished templates; this is the executable pilot.

In [ ]:
# Check the active kernel before importing the project, so setup errors are clear.
import importlib.util
import sys
print("Active Python:", sys.executable)
if sys.version_info < (3, 11):
    raise RuntimeError("Select a Python 3.11+ project .venv kernel.")
missing = [name for name in ("reddit_sna", "matplotlib", "numpy", "igraph", "leidenalg")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing {missing} in {sys.executable}. Select the project .venv kernel. "
        'In a terminal, use that Python to run: python -m pip install -e ".[graph,notebook]". '
        "Then restart the kernel and Run All."
    )
import os
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display
from reddit_sna.config import load_config
from reddit_sna.pipelines.preparation import run_preparation
from reddit_sna.reporting.pilot import load_pilot
from reddit_sna.reporting.figures import build_figures, export_figures
import matplotlib.pyplot as plt


## Input selection and replay versus rebuild

Paths are resolved from the working directory or `REDDIT_SNA_PROJECT`; they never depend on a personal absolute path. The default detection run below is the documented pilot, not an automatic choice of the best result. Its graph and normalization sources are followed from recorded manifests. To select another run, edit `DETECTION_RUN` or set `REDDIT_DETECTION_RUN`; setting it to `None` requires exactly one candidate.

**REPLAY** checks the existing data-to-membership chain, including every recorded artifact hash. It does not scan raw data again. **REBUILD** scans all 15 original ZIPs and runs audit, normalization, snapshots and detection using the same API as the CLI. It takes substantially longer and requires local raw data. Every rebuild/export uses a fresh output ID; existing runs remain immutable.

In [ ]:
# Search upward because editors may start the kernel in the notebook folder.
search = Path(os.environ.get("REDDIT_SNA_PROJECT", Path.cwd())).resolve()
ROOT = next((p for p in (search, *search.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/reddit_sna").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Set REDDIT_SNA_PROJECT to the repository directory and rerun.")
REBUILD = False
DETECTION_RUN = os.environ.get("REDDIT_DETECTION_RUN", "reddit-pilot-detection-v1")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
if REBUILD:
    # All stages share one new run; no raw archives are extracted or changed.
    rebuilt = run_preparation(load_config(ROOT / "configs/core.toml", run_id=f"reddit-pilot-{STAMP}"))
    DETECTION_RUN = rebuilt.name
pilot = load_pilot(ROOT, DETECTION_RUN)
print("Mode:", "REBUILD RAW" if REBUILD else "REPLAY VERIFIED ARTIFACTS")
print("Verified runs:", {stage: path.name for stage, path in pilot["runs"].items()})
print("Observed interval:", pilot["graphs"]["start_month"], "to", pilot["graphs"]["end_month_exclusive"], "(exclusive, UTC)")
print("Graph reference:", pilot["graphs"]["graph_reference"])
print("Producer versions:", pilot["detection"]["versions"])


## Graph semantics and descriptive evaluation

A directed reply links the comment author to the parent author at the child's timestamp. Parents may precede the pilot but cannot follow the child. Self-replies and excluded authors do not form edges. The undirected projection sums both directions: `R_uv = replies_uv + replies_vu`. Only endpoint users receive memberships; observed isolates are retained in coverage counts.

Independent Leiden optimizes weighted RBConfiguration at the recorded gamma separately for each month and seed. The first configured seed is canonical; it is not selected by its score. Community IDs are local to a month, and users are comparable only inside the recorded identity scope. Modularity measures graph quality; pairwise seed ARI measures repeatability. Neither establishes social validity or forecasting accuracy.

In [ ]:
# Show aggregate diagnostics; never display author names, discussion text or user IDs.
graphs, detection = pilot["graphs"], pilot["detection"]
for graph, result in zip(graphs["months"], detection["months"]):
    canonical = next(r for r in result["runs"] if r["seed"] == detection["canonical_seed"])
    print({"month": graph["month"], "reply_events": graph["reply_events"],
           "active_authors": graph["active_authors"], "reply_endpoints": graph["reply_endpoints"],
           "active_coverage": graph["reply_coverage_active_authors"],
           "communities": canonical["communities"], "modularity": canonical["modularity"],
           "seed_pair_ARI": result["seed_pair_ARI"]})


## Figures 1–2: data scope and corpus support

Raw scanned rows, retained pre-cutoff metadata, historical parents and pilot activity are overlapping populations. Figure 1 uses a labeled symlog axis to keep them readable; historical metadata is not a preprocessing loss. Figure 2 retains every declared corpus-month, with counts annotated and color `log(1 + records)`; **zero observed activity does not prove a corpus did not exist**.

In [ ]:
# Build all figures from verified summaries; full-graph metrics are never estimated from the display sample.
SAMPLE_LIMIT, DISPLAY_SEED = 150, 42
plt.ioff()  # Display each figure explicitly once; avoid duplicate inline output.
with plt.rc_context({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False}):
    figures = build_figures(pilot, sample_limit=SAMPLE_LIMIT, seed=DISPLAY_SEED)
# Detach figures from pyplot auto-display; the retained objects still display/export.
for fig in figures.values():
    plt.close(fig)
for name in ("01_data_scope", "02_corpus_support"):
    display(figures[name])


## Figures 3–5: graph volume, coverage and exclusive reply outcomes

Active authors and reply endpoints are distinct: endpoints can include historical parent-only users. Observed users are their union. Active coverage is covered active authors / active authors; observed coverage is endpoints / observed users. Undefined ratios remain missing, not zero.

Reply outcomes use the fixed priority: no pointer, missing parent, excluded child author, excluded parent author, future parent, self-reply, valid. They sum to comments. Older incidence counters can overlap and must not be stacked as losses. Unique authors across corpora are also not additive.

In [ ]:
for name in ("03_graph_volume", "04_coverage", "05_reply_outcomes"):
    display(figures[name])


## Figures 6–8: community sizes, repeatability and a network illustration

The empirical size distribution uses the full canonical partition and a logarithmic size axis; empty partitions are omitted. Quality and ARI panels show recorded diagnostics without confidence intervals: seed pairs are dependent repeats, not independent test sets.

The network figure shows up to 150 nodes reached by seeded BFS in the first month's largest component, with induced edges and full-graph community colors. It is a **biased illustration**, not a representative sample. Colors can repeat and have no temporal alignment. No user labels are exposed; the unweighted layout does not affect detection.

In [ ]:
for name in ("06_community_sizes", "07_partition_diagnostics", "08_network_sample"):
    display(figures[name])


## Export and handoff

Export all eight figures as PNG (180 dpi) and SVG under a new local artifact directory, alongside source-manifest hashes, plotting versions, graph reference and sampling policy. Keep generated outputs out of Git. Save an executed copy of this notebook inside the same run folder through the editor if needed; the tracked notebook remains output-free.

In [ ]:
# Fresh directories prevent a later Run All from silently replacing published figures.
FIGURE_OUTPUT = ROOT / "artifacts/core" / f"reddit-pilot-figures-{STAMP}"
export_figures(figures, FIGURE_OUTPUT, pilot, sample_limit=SAMPLE_LIMIT, seed=DISPLAY_SEED)
print("Figures and provenance:", FIGURE_OUTPUT)
for fig in figures.values():
    plt.close(fig)


## Limitations and next experiments

Three months establish a working data-to-membership pilot, not forecasting evidence or paper readiness. Corpus imbalance and missing history affect interpretation. Pseudonyms do not guarantee anonymity; additional bots are not comprehensively excluded.

Next steps are difficult planted benchmarks, gamma/perturbation sensitivity, a longer train-only pilot, validated tracking, cutoff-safe features and out-of-time prediction baselines. Keep alternate detector/tracker references separate and report null results. `evaluation.causality.compare_causal_prefix` checks aligned past graph/partition invariance; it does not validate unimplemented forecasting.

Handoff files: `memberships/<month>/membership-seed0.csv`, matching `snapshots/<month>/{R,directed_R,users}.csv` and their manifests. Never join cluster names across months or identities across normalized runs without a verified alignment.